In [1]:
import os
from datasets import load_dataset, Dataset, load_from_disk
import re
from collections import Counter

/opt/homebrew/anaconda3/envs/buildinglargelanguagemodel/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Loading the Pre-Train Dataset

In [2]:
dataset_dict = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", streaming=True)

In [3]:
dataset = dataset_dict['train'].take(10000)

In [4]:
sample_batch = dataset.take(10)

for _, row in enumerate(sample_batch):
    print(row['text'][:500])

The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.
Elizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.
The freedom she exhibited in finally accepting him in direct defiance of Lady Cath
Taking Play Seriously
By ROBIN MARANTZ HENIG
Published: February 17, 2008
On a drizzly Tuesday night in late January, 200 people came out to hear a psychiatrist talk rhapsodically about play -- not just the intense, joyous play of children, but play for all people, at all ages, at all times. (All species too; the lecture featured touching photos of a polar bear and a husky engaging playfully at a snowy outpost in northern Canada.) Stuart Brown, president of the National Institute for Play, was 

In [5]:
local_data_list = list(dataset_dict['train'].take(10000))

In [6]:
local_dataset = Dataset.from_list(local_data_list)

In [7]:
if not os.path.exists("local_fineweb_data"):
    local_dataset.save_to_disk("local_fineweb_data/hf_format")

In [8]:
local_dataset = load_from_disk("local_fineweb_data/hf_format")

In [9]:
for index in range(10):
    print(local_dataset[index]['text'][:500])

The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.
Elizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.
The freedom she exhibited in finally accepting him in direct defiance of Lady Cath
Taking Play Seriously
By ROBIN MARANTZ HENIG
Published: February 17, 2008
On a drizzly Tuesday night in late January, 200 people came out to hear a psychiatrist talk rhapsodically about play -- not just the intense, joyous play of children, but play for all people, at all ages, at all times. (All species too; the lecture featured touching photos of a polar bear and a husky engaging playfully at a snowy outpost in northern Canada.) Stuart Brown, president of the National Institute for Play, was 

In [10]:
total_tokens = sum(local_dataset['token_count'])
print(f"the training data has {total_tokens} tokens")

the training data has 10302766 tokens


**REGEX PATTERN And Tokenization By Row**

In [11]:
REGEX_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?[a-zA-Z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+(?!\S)|\s+"""

In [12]:
tokenizer_regex = re.compile(REGEX_PATTERN)

In [13]:
def regex_tokenized(text):
    return re.findall(tokenizer_regex, text)

In [14]:
def process_row(row):
    raw_text_string = row['text']
    row['string_tokens'] = regex_tokenized(raw_text_string)
    return row

In [15]:
tokenized_data_set = local_dataset.map(process_row, num_proc=10)

In [16]:
first_doc_tokens = tokenized_data_set[1]['string_tokens']
for _ , token in enumerate(first_doc_tokens[:30]):
    print(token)


Taking
 Play
 Seriously


By
 ROBIN
 MARANTZ
 HENIG


Published
:
 February
 17
,
 2008


On
 a
 drizzly
 Tuesday
 night
 in
 late
 January
,
 200
 people
 came
 out
 to


### Building the Vocabulary

In [17]:
token_counter = Counter()

In [25]:
for row in tokenized_data_set:
    token_counter.update(row['string_tokens'])

special_tokens = ["<|unk|>", "<|endoftext|>"]

all_tokens = [tok for tok,  _ in  token_counter.most_common()]

vocab_tokens = special_tokens + sorted(all_tokens)

token_to_id = {tok: idx for idx, tok in enumerate(vocab_tokens)}
id_to_token = {idx: tok for tok, idx in token_to_id.items()}

vocab_size = len(token_to_id)

In [26]:
vocab_tokens[:10]

['<|unk|>',
 '<|endoftext|>',
 '\n',
 ' ',
 ' \n',
 '  ',
 '   ',
 ' !',
 ' !!',
 ' !!!']

In [27]:
UNK_ID = token_to_id["<|unk|>"]
EOT_ID = token_to_id["<|endoftext|>"]

In [28]:
def tokens_to_ids(row):
    row['token_ids'] = [token_to_id.get(tok, UNK_ID) for tok in row['string_tokens']]
    return row

In [33]:
tokenized_with_ids = tokenized_data_set.map(tokens_to_ids, num_proc=10)

In [36]:
first_row = tokenized_with_ids[0]

for token, token_id in zip(first_row['string_tokens'][:10], first_row['token_ids'][:10]):
    print(f"{token!r:20} -> {token_id}")

'The'                -> 196119
' Independent'       -> 40565
' Jane'              -> 42280
'\n'                 -> 2
'For'                -> 178177
' all'               -> 90083
' the'               -> 152473
' love'              -> 125186
','                  -> 162172
' romance'           -> 143378
